---
## 0. Setup

In [1]:
%pip install -q -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import os

os.environ.setdefault("CHUNK_TOKEN_TARGET", "1024")
os.environ.setdefault("FIGURE_AREA_THRESHOLD", "0.01")
os.environ.setdefault("REPORT_DIR", "reports")


'reports'

In [4]:
import _json
from pathlib import Path

import pandas as pd

from rag import chunking, clients, config, docling_io, tables
from rag import index as index_module
from rag import inspect as inspection
from rag import sync as sync_module

SOURCE_PDF = Path("pdfs/clinical_trials_20/NCT03164772_NSCLC_mRNA_Vaccine.pdf")

BUCKET = None

DOC_ID = config.slugify(SOURCE_PDF.stem)

print(f"document    : {DOC_ID}")
print(f"embedding   : {config.EMBED_MODEL} ({clients.EMBED_DIMS}d)")
print(f"chunk size  : {config.CHUNK_TOKENS} tokens")
print(f"vision      : {config.VISION_MODEL}")
print(f"report      : {config.REPORT_DIR.resolve()}")

document    : nct03164772-nsclc-mrna-vaccine
embedding   : text-embedding-3-small (1536d)
chunk size  : 1024 tokens
vision      : gpt-4o-mini
report      : C:\Users\yogav\PycharmProjects\langchain-agents\05-rag\reports


In [5]:
doc = docling_io.parse_pdf(SOURCE_PDF)

  enrichments ON : do_code_enrichment, do_formula_enrichment, do_ocr, do_picture_classification, generate_picture_images, generate_table_images
  enrichments OFF: do_chart_extraction, do_picture_description, enable_remote_services
  (do_picture_description and enable_remote_services are off on purpose — describe_figures() runs after the parse, cached)
  heading hierarchy: ENABLED — SectionHeaderItem.level will be rewritten from bookmarks, then numbering, then style. This changes what a heading path means for every downstream comparison; re-check headings.py's rules and chunking.py's merge behaviour against the new output before trusting it.
  pipeline options in effect:
    do_ocr                        True
    do_table_structure            True
    do_picture_classification     True
    do_formula_enrichment         True
    do_code_enrichment            True
    generate_picture_images       True
    generate_table_images         True
    do_picture_description        False
    enab

Loading weights:   0%|          | 0/360 [00:00<?, ?it/s]

[INFO] 2026-09-12 15:13:12,847 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-12 15:13:12,860 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-12 15:13:12,886 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-12 15:13:12,887 [RapidOCR] main.py:50: Using C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.pth
[INFO] 2026-09-12 15:13:13,033 [RapidOCR] base.py:23: Using engine_name: torch
[INFO] 2026-09-12 15:13:13,034 [RapidOCR] device_config.py:57: Using CPU device
[INFO] 2026-09-12 15:13:13,039 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\rapidocr\models\ch_ptocr_mobile_v2.0_cls_mobile.pth
[INFO] 2026-09-12 15:13:13,040 [RapidOCR] main.py:50: Using C:\Users\yogav\PycharmProjects\langcha

Loading weights:   0%|          | 0/770 [00:00<?, ?it/s]

[transformers] Model config: pad_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 128002. This may result in unexpected behavior.


Loading weights:   0%|          | 0/471 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie model.text_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.
C:\Users\yogav\PycharmProjects\langchain-agents\.venv\Lib\site-packages\torch\nn\modules\conv.py:560: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\Convolution.cpp:1105.)
  return F.conv2d(
Output queue closed while emitting from layout
Output queue closed while emitting from layout
Output queue closed while emitting from layout
Output queue closed while emitting from layout
Output queue closed while emitting from layout
Output queue closed while emitting from layout
Output queue closed while emitting from lay

KeyboardInterrupt: 

In [ ]:
markdown = doc.export_to_markdown()
doc_date = chunking.document_date(SOURCE_PDF, markdown[:4000])

print(f"pages {len(doc.pages)}  document date: {doc_date}")

In [ ]:
from rag.headings import clean_headings

clean_headings(doc)

In [ ]:
from backup.RAG.aws.rag.config import CHUNK_TOKENS
from rag.config import ENCODING
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.hierarchical_chunker import (
    ChunkingDocSerializer, ChunkingSerializerProvider,TripletTableSerializer
)

from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from docling_core.transforms.serializer.markdown import MarkdownDocSerializer

class MarkdownTableProvider(ChunkingSerializerProvider):
    def get_serializer(self, doc, **kwargs):
        return ChunkingDocSerializer(
            doc=doc, table_serializer=TripletTableSerializer()
        )

In [ ]:
chunker = HybridChunker(
    tokenizer=OpenAITokenizer(tokenizer=ENCODING, max_tokens=CHUNK_TOKENS),

    serializer_provider=MarkdownTableProvider(),

    merge_peers=False,
)

In [ ]:
chunks = list(chunker.chunk(dl_doc=doc))

In [ ]:
uris = docling_io.save_figures(doc, DOC_ID, BUCKET)

print(f"{len(uris)} figure images stored")

In [ ]:
from rag.chunking import _to_entries

"""
chunks
   ↓
HybridChunker output

chunker
   ↓
used to recreate contextualized text

figure_uris
   ↓
mapping of Docling picture refs → stored image URI
"""

entries, dropped = _to_entries(chunks, chunker, uris)

In [ ]:
from rag.chunking import _merge_prose

entries, merges = _merge_prose(entries)

In [ ]:
from rag.chunking import _apply_floor

entries, floor_merges = _apply_floor(entries)

In [ ]:
from rag.chunking import _to_records, _record_factory

items_by_ref = {}

for item, _ in doc.iterate_items():
    ref = getattr(item, "self-ref", None)

    if ref:
        items_by_ref[ref] = item

make = _record_factory(SOURCE_PDF, DOC_ID, doc_date)

records, table_groups, fig_stats = _to_records(
    entries, doc,  items_by_ref, uris, make
)

In [ ]:
print(records[0])
print("-------------")
for record in records:
    if record['meta']['content_type'] == 'figure':
        print(record)
        break
print("-------------")
print(records[5])
print("-------------")
print(records[4])

In [ ]:
from rag.tables import table_markdown
from rag.chunking import _table_summaries

tables = table_markdown(doc)

summaries, table_stats = _table_summaries(
    table_groups, tables, doc, items_by_ref, make
)

records += summaries

In [ ]:
from rag.chunking import _finalise

records, truncated = _finalise(records)

In [ ]:
from rag.chunking import _report

_report(
    records,
    chunks,
    tables,
    table_groups,
    {
        **fig_stats,
        **table_stats,
        "dropped": dropped,
        "merges": merges,
        "floor_merges": floor_merges,
        "truncated": truncated,
    }
)

In [ ]:
from rag import index as index_module
from rag import sync as sync_module

index = index_module.open_index(create=True)
plan = sync_module.sync(index, DOC_ID, records)
print(plan)